In [5]:
pip install rasterio fiona geopandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
cut_area.py
===========
Módulo de recorte espacial em lote de rasters GeoTIFF pela área de estudo
definida em um shapefile (limite do Pantanal).

Fluxo principal:
    1. Carregamento do shapefile de limite (uma única vez).
    2. Reprojeção do shapefile para o CRS do raster, se necessário
       (feita uma vez antes do loop, assumindo CRS consistente entre rasters).
    3. Recorte (clip) de cada raster pelas geometrias do shapefile.
    4. Substituição do valor nodata original por NaN.
    5. Exportação como GeoTIFF float32 com compressão LZW.

Convenção de nodata:
    O valor nodata original do arquivo é substituído por NaN após o recorte.
    Pixels com valor 0 legítimo (ex.: radiância zero) são preservados.

Dependências:
    rasterio, geopandas, numpy

Uso típico:
    process_raster_batch(
        input_dir='MASCARA_APLICADA',
        shapefile_path='Limite_Pantanal/biome_border.shp',
        output_dir='CUT',
    )
"""

import os
from pathlib import Path
from typing import List, Optional

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.mask import mask as rasterio_mask

# Fallback para tqdm: se não instalado, usa iterador simples com print
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, desc="", **kwargs):
        """Substituto simples de tqdm quando a biblioteca não está instalada."""
        print(f"{desc}..." if desc else "Processando...")
        return iterable

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Prefixo adicionado ao nome dos arquivos recortados
OUTPUT_PREFIX = "clip_"

# dtype de saída: consistente com os demais módulos do pipeline
OUTPUT_DTYPE = "float32"


# ---------------------------------------------------------------------------
# Funções auxiliares (uso interno)
# ---------------------------------------------------------------------------

def _load_and_reproject_shapefile(
    shapefile_path: str,
    target_crs,
) -> gpd.GeoDataFrame:
    """
    Carrega um shapefile e o reprojeta para o CRS alvo, se necessário.

    Parâmetros:
        shapefile_path (str): Caminho do arquivo shapefile.
        target_crs:           CRS de destino (objeto rasterio/pyproj CRS).

    Retorna:
        GeoDataFrame: Shapefile no CRS do raster de destino.

    Levanta:
        FileNotFoundError: Se o shapefile não existir.
    """
    if not os.path.exists(shapefile_path):
        raise FileNotFoundError(f"Shapefile não encontrado: '{shapefile_path}'")

    gdf = gpd.read_file(shapefile_path)

    if gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)

    return gdf


def _clip_raster(
    src: rasterio.DatasetReader,
    shapes: list,
) -> tuple:
    """
    Recorta um raster aberto pelas geometrias fornecidas.

    Parâmetros:
        src    (DatasetReader): Raster aberto via rasterio.
        shapes (list):          Lista de geometrias shapely para o recorte.

    Retorna:
        tuple: (out_image, out_transform) — array recortado e nova transformada.
    """
    return rasterio_mask(src, shapes, crop=True)


def _replace_nodata_with_nan(
    data: np.ndarray,
    nodata_value,
) -> np.ndarray:
    """
    Substitui o valor nodata original do raster por NaN.

    Substitui apenas o nodata declarado no arquivo — pixels com valor 0
    legítimo (ex.: radiância zero) são preservados.
    Se o arquivo não declarar nodata, retorna o array sem alteração.

    Parâmetros:
        data         (np.ndarray): Array de dados do raster (float32).
        nodata_value:              Valor nodata original (pode ser None).

    Retorna:
        np.ndarray: Array com nodata substituído por NaN.
    """
    if nodata_value is not None:
        data = np.where(data == nodata_value, np.nan, data)
    return data


# ---------------------------------------------------------------------------
# Função pública
# ---------------------------------------------------------------------------

def process_raster_batch(
    input_dir: str,
    shapefile_path: str,
    output_dir: str,
    output_prefix: str = OUTPUT_PREFIX,
) -> int:
    """
    Recorta em lote arquivos GeoTIFF de um diretório usando um shapefile de limite.

    O shapefile é carregado e reprojetado uma única vez antes do loop,
    assumindo que todos os rasters do diretório compartilham o mesmo CRS
    (garantido pelo pipeline anterior).

    Parâmetros:
        input_dir      (str): Diretório contendo os arquivos `.tif` de entrada.
        shapefile_path (str): Caminho do shapefile de limite da área de estudo.
        output_dir     (str): Diretório de saída para os rasters recortados.
        output_prefix  (str): Prefixo adicionado ao nome dos arquivos de saída
                              (padrão: 'clip_').

    Retorna:
        int: Número de arquivos processados com sucesso.

    Levanta:
        FileNotFoundError: Se `input_dir` ou `shapefile_path` não existirem.
    """
    # Valida diretório de entrada
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Diretório de entrada não encontrado: '{input_dir}'")

    # Lista arquivos .tif ordenados para comportamento determinístico
    tif_files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".tif")
    )

    if not tif_files:
        print(f"⚠️  Nenhum arquivo .tif encontrado em '{input_dir}'.")
        return 0

    os.makedirs(output_dir, exist_ok=True)

    # Lê o CRS do primeiro raster para reprojetar o shapefile uma única vez
    first_raster = os.path.join(input_dir, tif_files[0])
    with rasterio.open(first_raster) as src:
        raster_crs = src.crs

    # Carrega e reprojeta o shapefile antes do loop
    gdf = _load_and_reproject_shapefile(shapefile_path, raster_crs)
    shapes = list(gdf.geometry)

    print(f"📁 Entrada  : {input_dir}")
    print(f"📁 Saída    : {output_dir}")
    print(f"🗺️  Shapefile: {shapefile_path}")
    print(f"📊 Arquivos : {len(tif_files)}\n")

    success = 0
    errors: List[str] = []

    for tif_file in tqdm(tif_files, desc="Recortando rasters"):
        raster_path = os.path.join(input_dir, tif_file)
        output_path = os.path.join(output_dir, f"{output_prefix}{tif_file}")

        try:
            with rasterio.open(raster_path) as src:
                nodata_original = src.nodata

                out_image, out_transform = _clip_raster(src, shapes)
                out_meta = src.meta.copy()

            # Converte para float32 e substitui nodata original por NaN
            out_image = out_image.astype(np.float32)
            out_image = _replace_nodata_with_nan(out_image, nodata_original)

            # Atualiza metadados com dimensões do recorte e otimizações de saída
            out_meta.update({
                "driver":    "GTiff",
                "height":    out_image.shape[-2],   # dimensão Y (banda, Y, X)
                "width":     out_image.shape[-1],   # dimensão X
                "transform": out_transform,
                "dtype":     OUTPUT_DTYPE,
                "nodata":    np.nan,
                "compress":  "lzw",
                "tiled":     True,
            })

            with rasterio.open(output_path, "w", **out_meta) as dst:
                dst.write(out_image)

            success += 1

        except Exception as e:
            errors.append(tif_file)
            print(f"\n  ✘ Erro em '{tif_file}': {e}")
            continue

    # Resumo final
    print(f"\n{'=' * 50}")
    print(f"✅ Sucesso : {success}/{len(tif_files)}")
    if errors:
        print(f"❌ Erros   : {len(errors)}/{len(tif_files)}")
        for name in errors:
            print(f"   • {name}")
    print(f"💾 Resultados em: {output_dir}")
    print("=" * 50)

    return success


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    INPUT_DIR      = "MASCARA_APLICADA"
    SHAPEFILE_PATH = os.path.join("Limite_Pantanal", "biome_border.shp")
    OUTPUT_DIR     = "CUT"

    process_raster_batch(
        input_dir=INPUT_DIR,
        shapefile_path=SHAPEFILE_PATH,
        output_dir=OUTPUT_DIR,
    )

📁 Entrada  : MASCARA_APLICADA
📁 Saída    : CUT
🗺️  Shapefile: Limite_Pantanal\biome_border.shp
📊 Arquivos : 60



Recortando rasters: 100%|██████████| 60/60 [00:04<00:00, 13.97it/s]


✅ Sucesso : 60/60
💾 Resultados em: CUT
